# GeoLift-S3 Lite - TAR2000 train 1,600 / val 400 / test 1,000

This notebook benchmarks S3 with the exact KITTI TAR2000 samples, splits, and protocol lock used by S2.

```text
MyDrive/GeoLift_Data/teacher_subset_2000/
|-- selected_2000_ids.json
`-- kitti_trainval_2000.tar
```

The initial S3 run is teacher-free: it does not extract metric or geometry teacher TARs. The RGB-only MobileNetV4-Conv-Small-0.5 loads ImageNet pretrained weights; sparse depth uses a separate tiny branch. Metric, log, sparse, and edge losses are active from the first epoch.

## Before Run all

Select a GPU runtime. Inputs use the same Drive directory as S2. S3 outputs and checkpoints are isolated; this notebook can resume only its own S3 checkpoint.

In [ ]:
# 0) Mount Drive and configure the S3 run
from google.colab import drive
from pathlib import Path
import os, sys, json, time, shutil, tarfile, zipfile, hashlib, subprocess
import numpy as np
import pandas as pd
import torch

drive.mount("/content/drive")
LOCAL_REPO = Path("/content/GeoDistill_RT")
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/GeoLift_Data")
TAR2000_ROOT = DRIVE_DATA_ROOT / "teacher_subset_2000"
FINAL_MANIFEST = TAR2000_ROOT / "selected_2000_ids.json"
KITTI_BUNDLE_TAR = TAR2000_ROOT / "kitti_trainval_2000.tar"
EXPECTED_KITTI_BUNDLE_BYTES = 2_041_970_176

DATA_ROOT = Path("/content/geolift_s3_data")
TEACHER_ROOT = DATA_ROOT / "teacher_outputs_unused"
KITTI_ROOT = DATA_ROOT / "kitti_bundle"
SPLIT_ROOT = DATA_ROOT / "splits"
RUN_LOCAL = Path("/content/geolift_s3_lite_run")
RUN_DRIVE = Path("/content/drive/MyDrive/GeoLift_RT_runs/v3_s3_lite_pretrained_train1600_val400")
S2_RUN_DRIVE = Path("/content/drive/MyDrive/GeoLift_RT_runs/v2_1_balancedloss_ablation_train1600_val400")
RESUME_S3 = True

FINAL_COUNT, TRAIN_COUNT, VAL_COUNT, TEST_COUNT = 2000, 1600, 400, 1000
EPOCHS, BATCH_SIZE, NUM_WORKERS = 30, 2, 2
for path in (DATA_ROOT, TEACHER_ROOT, SPLIT_ROOT, RUN_LOCAL, RUN_DRIVE):
    path.mkdir(parents=True, exist_ok=True)
for path in (FINAL_MANIFEST, KITTI_BUNDLE_TAR):
    assert path.is_file(), f"Missing input: {path}"
assert KITTI_BUNDLE_TAR.stat().st_size == EXPECTED_KITTI_BUNDLE_BYTES
assert torch.cuda.is_available(), "Select a GPU runtime before Run all."
free_gib = shutil.disk_usage("/content").free / 1024**3
required_gib = KITTI_BUNDLE_TAR.stat().st_size / 1024**3 + 12.0
assert free_gib >= required_gib, (free_gib, required_gib)
print("GPU:", torch.cuda.get_device_name(0))
print("Input:", FINAL_MANIFEST, KITTI_BUNDLE_TAR)
print("S3 output:", RUN_DRIVE)
print(f"Local SSD: {free_gib:.1f} GiB; required about {required_gib:.1f} GiB")

## 1. Clone remote/main to Colab SSD

Clone source directly from GitHub so large teacher weights on Drive are never copied.

In [ ]:
# 1) Clone source trực tiếp từ GitHub vào SSD Colab
from pathlib import Path
import shutil
import subprocess
import sys
import os

GITHUB_REPO = "https://github.com/PhuocDang2104/GeoDistill_RT.git"
LOCAL_REPO = Path("/content/GeoDistill_RT")

# Xóa bản copy dở từ lần chạy lỗi
if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)

# Lấy đúng source mới nhất từ remote/main
subprocess.run(
    [
        "git",
        "clone",
        "--depth", "1",
        "--branch", "main",
        GITHUB_REPO,
        str(LOCAL_REPO),
    ],
    check=True,
)

os.chdir(LOCAL_REPO)

# Kiểm tra source bắt buộc
for required in ("src", "scripts", "configs", "tests"):
    path = LOCAL_REPO / required
    assert path.exists(), f"Repo thiếu: {path}"

# Cài dependencies
requirements = LOCAL_REPO / "requirements.txt"
if requirements.is_file():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(requirements),
        ],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(LOCAL_REPO), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Clone source thành công")
print("Local repo:", LOCAL_REPO)
print("Source commit:", commit)
print("Python:", sys.version)

## 2. Lock the 1,600/400 manifest

Verify sample IDs and raw-drive disjointness against the same benchmark manifest; teacher TARs are not opened.

In [ ]:
# 2) Manifest gate - same samples as S2
def raw_drive(sample_id: str) -> str:
    return sample_id.split("_sync_image_")[0]

final_manifest = json.loads(FINAL_MANIFEST.read_text(encoding="utf-8"))
train_ids = list(final_manifest["train_ids"])
val_ids = list(final_manifest["val_ids"])
selected_ids = train_ids + val_ids
assert len(train_ids) == TRAIN_COUNT
assert len(val_ids) == VAL_COUNT
assert len(set(selected_ids)) == FINAL_COUNT
assert not (set(train_ids) & set(val_ids))
train_drives = {raw_drive(item) for item in train_ids}
val_drives = {raw_drive(item) for item in val_ids}
assert not (train_drives & val_drives)
dataset_manifest = RUN_LOCAL / "s3_dataset_manifest.json"
dataset_manifest.write_text(json.dumps({
    "train_count": TRAIN_COUNT,
    "val_count": VAL_COUNT,
    "train_ids": train_ids,
    "val_ids": val_ids,
    "train_drives": sorted(train_drives),
    "val_drives": sorted(val_drives),
    "teacher_used": False,
}, indent=2), encoding="utf-8")
print("Manifest gate passed:", TRAIN_COUNT, VAL_COUNT, "drive overlap=0")

## 3. Extract the KITTI bundle and official test 1,000

This keeps the same data contract as the S2 notebook.

In [ ]:
# 3) Safe-extract KITTI train/val bundle, sau đó lấy official test selection
def safe_extract_tar(archive_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with tarfile.open(archive_path, "r:*") as archive:
        members = archive.getmembers()
        for member in members:
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"Unsafe TAR member: {member.name}")
        archive.extractall(destination, members=members)


def bundle_is_ready() -> bool:
    report_path = KITTI_ROOT / "kitti_bundle_report.json"
    if not report_path.is_file():
        return False
    try:
        report = json.loads(report_path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return (
        report.get("contract_ok") is True
        and report.get("selected_count") == FINAL_COUNT
        and report.get("train_count") == TRAIN_COUNT
        and report.get("val_count") == VAL_COUNT
        and report.get("roles", {}).get("rgb", {}).get("count") == FINAL_COUNT
        and report.get("roles", {}).get("sparse", {}).get("count") == FINAL_COUNT
        and report.get("roles", {}).get("groundtruth", {}).get("count") == FINAL_COUNT
        and report.get("roles", {}).get("intrinsics", {}).get("count") == FINAL_COUNT
    )


if not bundle_is_ready():
    if KITTI_ROOT.exists():
        shutil.rmtree(KITTI_ROOT)
    safe_extract_tar(KITTI_BUNDLE_TAR, KITTI_ROOT)

assert bundle_is_ready(), "KITTI bundle report không đạt contract."
bundle_report_path = KITTI_ROOT / "kitti_bundle_report.json"
bundle_report = json.loads(bundle_report_path.read_text(encoding="utf-8"))
bundle_manifest_path = KITTI_ROOT / "selected_2000_ids.json"
assert bundle_manifest_path.is_file(), bundle_manifest_path
bundle_manifest = json.loads(bundle_manifest_path.read_text(encoding="utf-8"))
assert bundle_manifest["train_ids"] == final_manifest["train_ids"], "Bundle/teacher train IDs không khớp"
assert bundle_manifest["val_ids"] == final_manifest["val_ids"], "Bundle/teacher val IDs không khớp"
assert hashlib.sha256(bundle_manifest_path.read_bytes()).hexdigest() == bundle_report["selected_manifest_sha256"]

# Official anonymous test is a separate 1,000-image, no-GT benchmark split.
test_root = KITTI_ROOT / "test_1000"
test_split_source = KITTI_ROOT / "splits" / "test_1000.txt"


def official_test_is_ready() -> bool:
    counts = (
        len(list((test_root / "image").glob("*.png"))),
        len(list((test_root / "velodyne_raw").glob("*.png"))),
        len(list((test_root / "intrinsics").glob("*.txt"))),
    )
    if counts != (TEST_COUNT, TEST_COUNT, TEST_COUNT) or not test_split_source.is_file():
        return False
    return len([line for line in test_split_source.read_text(encoding="utf-8").splitlines() if line.strip()]) == TEST_COUNT


if not official_test_is_ready():
    if test_root.exists():
        shutil.rmtree(test_root)
    test_zip = DATA_ROOT / "data_depth_selection.zip"
    subprocess.run(
        [
            "wget", "-q", "--show-progress", "-c",
            "https://s3.eu-central-1.amazonaws.com/avg-kitti/data_depth_selection.zip",
            "-O", str(test_zip),
        ],
        check=True,
    )
    test_lines = []
    with zipfile.ZipFile(test_zip) as archive:
        names = set(archive.namelist())
        image_members = sorted(
            name for name in names
            if "/test_depth_completion_anonymous/image/" in "/" + name and name.endswith(".png")
        )
        assert len(image_members) == TEST_COUNT, len(image_members)
        for index, image_member in enumerate(image_members, start=1):
            filename = Path(image_member).name
            stem = Path(filename).stem
            base = image_member.rsplit("/image/", 1)[0]
            sparse_member = f"{base}/velodyne_raw/{filename}"
            intrinsics_member = f"{base}/intrinsics/{stem}.txt"
            assert sparse_member in names and intrinsics_member in names, stem
            destinations = (
                (image_member, test_root / "image" / filename),
                (sparse_member, test_root / "velodyne_raw" / filename),
                (intrinsics_member, test_root / "intrinsics" / f"{stem}.txt"),
            )
            for member, destination in destinations:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, destination.open("wb") as output:
                    shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
            test_lines.append(
                f"{stem} test_1000/image/{filename} test_1000/velodyne_raw/{filename} "
                f"none test_1000/intrinsics/{stem}.txt"
            )
            if index % 100 == 0:
                print(f"Official test extract: {index:,}/{TEST_COUNT:,}")
    test_split_source.parent.mkdir(parents=True, exist_ok=True)
    test_split_source.write_text("\n".join(test_lines) + "\n", encoding="utf-8")
    test_zip.unlink(missing_ok=True)

assert official_test_is_ready(), "Official KITTI test 1.000 chưa đầy đủ."

expected_splits = {
    "train_1600.txt": TRAIN_COUNT,
    "val_400.txt": VAL_COUNT,
    "test_1000.txt": TEST_COUNT,
}
for filename, expected in expected_splits.items():
    source = KITTI_ROOT / "splits" / filename
    assert source.is_file(), source
    shutil.copy2(source, SPLIT_ROOT / filename)
    lines = [line for line in source.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert len(lines) == expected, (filename, len(lines), expected)

    # Mọi path được loader sử dụng phải tồn tại trước khi train.
    for line in lines:
        tokens = line.split()
        assert len(tokens) in (5, 8), (filename, line)
        for token in tokens[1:3]:
            assert (KITTI_ROOT / token).is_file(), (filename, token)
        if tokens[3].lower() not in ("none", "null", "-"):
            assert (KITTI_ROOT / tokens[3]).is_file(), (filename, tokens[3])
        if len(tokens) == 5:
            assert (KITTI_ROOT / tokens[4]).is_file(), (filename, tokens[4])

print(json.dumps(bundle_report, indent=2))
print("KITTI splits OK:", expected_splits)

## 4. Inspect train/validation inputs

The S3 loader reads only RGB, sparse depth, GT, and intrinsics. Teacher keys must be absent.

In [ ]:
# 4) Dataset inspection without teacher I/O
import matplotlib.pyplot as plt
from src.dataset import KITTIDepthCompletionDataset

def inspect_split(split_name: str, split_file: str):
    dataset = KITTIDepthCompletionDataset(
        data_root=KITTI_ROOT, split_root=SPLIT_ROOT, split_file=split_file,
        split_name=split_name, image_size=(352, 1216), teacher_root=TEACHER_ROOT,
        load_teacher=False, load_geometry=False, load_mono=False, return_tensors=True,
    )
    sample = dataset[0]
    assert "D_cm" not in sample and "R_G" not in sample
    panels = [
        ("RGB", sample["rgb"].permute(1,2,0).numpy(), None),
        ("Sparse", sample["sparse"][0].numpy(), "turbo"),
        ("Sparse mask", sample["mask"][0].numpy(), "gray"),
        ("GT", sample["gt"][0].numpy(), "turbo"),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    for axis, (title, image, cmap) in zip(axes, panels):
        axis.imshow(image, cmap=cmap); axis.set_title(title); axis.axis("off")
    fig.suptitle(f"{split_name}: {sample['sample_id']}"); plt.tight_layout(); plt.show()
    return dataset

train_dataset = inspect_split("train", "train_1600.txt")
val_dataset = inspect_split("val", "val_400.txt")
assert len(train_dataset) == TRAIN_COUNT and len(val_dataset) == VAL_COUNT

## 5. Resolve the isolated S3 config

MobileNetV4 pretrained initialization is enabled. Teacher loading and curriculum-gated losses are disabled. Resume uses only the S3 run directory.

In [ ]:
# 5) Resolve isolated S3 config
import yaml
paths_file = DATA_ROOT / "paths_geolift_s3_tar2000.yaml"
paths = {
    "data_root": str(KITTI_ROOT), "split_root": str(SPLIT_ROOT),
    "train_split": "train_1600.txt", "val_split": "val_400.txt", "test_split": "test_1000.txt",
    "teacher_root": str(TEACHER_ROOT), "student_root": str(RUN_LOCAL),
}
paths_file.write_text(yaml.safe_dump(paths, sort_keys=False), encoding="utf-8")
base_config_path = LOCAL_REPO / "configs" / "geolift_s3_lite_tar2000.yaml"
assert base_config_path.is_file(), base_config_path
cfg = yaml.safe_load(base_config_path.read_text(encoding="utf-8"))
cfg["paths_file"] = str(paths_file)
cfg["train"]["epochs"] = EPOCHS
cfg["train"]["batch_size"] = BATCH_SIZE
cfg["data"]["num_workers"] = NUM_WORKERS
cfg["data"]["load_metric_teacher"] = False
cfg["outputs"]["backup_root"] = str(RUN_DRIVE)
cfg["train"]["resume"] = None
last_checkpoint = RUN_DRIVE / "checkpoints" / "last.pth"
if RESUME_S3 and last_checkpoint.exists():
    local_ckpt = RUN_LOCAL / "checkpoints" / "last.pth"
    local_ckpt.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(last_checkpoint, local_ckpt)
    cfg["train"]["resume"] = str(local_ckpt)
resolved_cfg = DATA_ROOT / "geolift_s3_lite_train1600_val400.yaml"
resolved_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
for source, name in [
    (resolved_cfg, "resolved_config.yaml"), (paths_file, "resolved_paths.yaml"),
    (FINAL_MANIFEST, "selected_2000_ids.json"), (dataset_manifest, "s3_dataset_manifest.json"),
    (bundle_report_path, "kitti_bundle_report.json"),
]:
    destination = RUN_LOCAL / name
    if source.resolve() != destination.resolve():
        shutil.copy2(source, destination)
run_manifest = {
    "architecture": "GeoLift-S3-Lite",
    "encoder": cfg["model"]["encoder"],
    "encoder_pretrained": cfg["model"]["encoder_pretrained"],
    "teacher_used": False,
    "loss_from_first_epoch": ["metric_multiscale", "log", "sparse_pre_anchor", "edge"],
    "train": TRAIN_COUNT, "val": VAL_COUNT, "test_anonymous": TEST_COUNT,
    "comparison_run": str(S2_RUN_DRIVE),
    "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
    "config_sha256": hashlib.sha256(resolved_cfg.read_bytes()).hexdigest(),
}
(RUN_LOCAL / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
print(resolved_cfg.read_text(encoding="utf-8"))

## 6. Unit tests and one-batch forward/backward

Gate initialization, tensor pyramid, hard anchor, and teacher-free loss must pass before training.

In [ ]:
# 6) Tests + real S3 smoke
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=LOCAL_REPO, check=True)
from src.utils import load_project_config
from src.train_student import make_loader
from src.model_factory import build_student
from src.losses import geort_loss
loaded_cfg, loaded_paths = load_project_config(resolved_cfg)
train_loader = make_loader(loaded_cfg, loaded_paths, "train", training=True)
batch = next(iter(train_loader))
assert "D_cm" not in batch and "R_G" not in batch and "C_G" not in batch
model = build_student(loaded_cfg).cuda().train()
gpu_batch = {key: (value[:1].cuda() if torch.is_tensor(value) else value) for key, value in batch.items()}
with torch.autocast("cuda", dtype=torch.float16):
    prediction = model(*(gpu_batch[key] for key in ("rgb","sparse","mask","ray","uv","K")))
    loss, items = geort_loss(prediction, gpu_batch, loaded_cfg["loss"], {}, 0, loaded_cfg["mono_ssi"])
assert torch.isfinite(loss)
assert items["w_metric"] == 1.0 and items["w_log"] == 0.2
assert items["w_sparse"] == 0.1 and items["w_edge"] == 0.05
for key in ("mean_residual_gate_8","mean_residual_gate_4","mean_residual_gate_2","mean_residual_gate_1"):
    assert 0.045 <= items[key] <= 0.055, (key, items[key])
identity_error = ((prediction["D_full"] - gpu_batch["sparse"]) * gpu_batch["mask"]).abs().max()
assert float(identity_error) == 0.0
loss.backward()
assert all(parameter.grad is None or torch.isfinite(parameter.grad).all() for parameter in model.parameters())
print("S3 smoke loss:", float(loss.detach()))
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Initial gates:", {k:v for k,v in items.items() if "gate" in k})
del model, prediction, batch, gpu_batch
train_loader._iterator = None
torch.cuda.empty_cache()

## 7. Train -> validation -> test -> FP16 profile

Use the same immutable protocol lock as S2. Checkpoints and logs are backed up to the S3 Drive directory after every epoch.

In [ ]:
# 9) Canonical train -> val -> anonymous test -> FP16 profile
PROTOCOL_LOCK = (
    DRIVE_DATA_ROOT
    / "protocols"
    / "kitti_tar2000_train1600_val400_test1000_v1.json"
)
PROTOCOL_LOCK.parent.mkdir(parents=True, exist_ok=True)
standard_command = [
    sys.executable,
    "scripts/run_standard_experiment.py",
    "--config",
    str(resolved_cfg),
    "--protocol-lock",
    str(PROTOCOL_LOCK),
    "--profile-warmup",
    "100",
    "--profile-runs",
    "500",
]
if not PROTOCOL_LOCK.exists():
    standard_command.append("--create-protocol-lock")
subprocess.run(standard_command, cwd=LOCAL_REPO, check=True)

subprocess.run(
    [
        "rsync",
        "-a",
        f"{RUN_LOCAL}/",
        f"{RUN_DRIVE}/",
    ],
    check=True,
)

best_checkpoint = RUN_LOCAL / "checkpoints" / "best.pth"
assert best_checkpoint.exists(), best_checkpoint

experiment_record = RUN_LOCAL / "experiment_record.json"
assert experiment_record.is_file(), experiment_record
record = json.loads(experiment_record.read_text(encoding="utf-8"))
assert record["status"] == "complete", record["status"]
print("Canonical run complete:", experiment_record)


## 8. Compare S3 with S2

Compare the best global validation RMSE checkpoints, FP16 efficiency, loss budget, and residual-gate dynamics.

In [ ]:
# 8) S3-vs-S2 benchmark tables and plots
import matplotlib.pyplot as plt
s2_log = S2_RUN_DRIVE / "logs" / "train_log.csv"
s3_log = RUN_LOCAL / "logs" / "train_log.csv"
assert s3_log.is_file(), s3_log
s3_df = pd.read_csv(s3_log)
if s2_log.is_file():
    s2_df = pd.read_csv(s2_log)
    s2_best = s2_df.loc[s2_df["val_rmse"].idxmin()]
    s3_best = s3_df.loc[s3_df["val_rmse"].idxmin()]
    metrics = [
        "val_rmse","val_mae","val_irmse","val_abs_rel","val_delta1",
        "val_rmse_0_20","val_rmse_20_40","val_rmse_40_60","val_rmse_60_80","val_rmse_80_120",
        "val_rmse_edge","val_rmse_nonedge",
    ]
    rows=[]
    for metric in metrics:
        old, new = float(s2_best[metric]), float(s3_best[metric])
        rows.append({"metric":metric,"S2_best":old,"S3_best":new,"S3_change_pct":100*(new-old)/max(abs(old),1e-12)})
    comparison=pd.DataFrame(rows)
    comparison.to_csv(RUN_LOCAL / "s3_vs_s2_metrics.csv", index=False)
    display(comparison)
    s2_profile_path = S2_RUN_DRIVE / "logs" / "geolift_component_profile.json"
    s3_profile_path = RUN_LOCAL / "logs" / "geolift_component_profile.json"
    if s2_profile_path.is_file() and s3_profile_path.is_file():
        s2_profile = json.loads(s2_profile_path.read_text(encoding="utf-8"))
        s3_profile = json.loads(s3_profile_path.read_text(encoding="utf-8"))
        efficiency_keys = ("total_median_ms", "total_p95_ms", "fps_from_median", "peak_allocated_mb", "model_parameters", "estimated_conv_linear_macs")
        efficiency = pd.DataFrame([
            {"architecture": "S2", **{key: s2_profile.get(key) for key in efficiency_keys}},
            {"architecture": "S3 Lite", **{key: s3_profile.get(key) for key in efficiency_keys}},
        ])
        efficiency.to_csv(RUN_LOCAL / "s3_vs_s2_efficiency.csv", index=False)
        display(efficiency)
    print("S2 best epoch:", int(s2_best["epoch"]), "| S3 best epoch:", int(s3_best["epoch"]))
    fig, axes = plt.subplots(1,3,figsize=(18,4))
    axes[0].plot(s2_df["epoch"],s2_df["val_rmse"],label="S2")
    axes[0].plot(s3_df["epoch"],s3_df["val_rmse"],label="S3 Lite")
    axes[0].set(xlabel="epoch",ylabel="val RMSE (m)",title="Convergence"); axes[0].grid(alpha=.3); axes[0].legend()
    labels=["0-20","20-40","40-60","60-80","80-120"]
    columns=[f"val_rmse_{x.replace('-','_')}" for x in labels]; pos=np.arange(5)
    axes[1].bar(pos-.2,[s2_best[c] for c in columns],.4,label="S2")
    axes[1].bar(pos+.2,[s3_best[c] for c in columns],.4,label="S3 Lite")
    axes[1].set_xticks(pos,labels); axes[1].set(xlabel="GT range (m)",ylabel="RMSE (m)",title="Depth ranges"); axes[1].legend()
else:
    print("S2 log not found; exporting S3 diagnostics only:", s2_log)
    fig, axes = plt.subplots(1,3,figsize=(18,4))
    axes[0].plot(s3_df["epoch"],s3_df["val_rmse"],label="S3 Lite"); axes[0].legend()

loss_parts = {
    "metric": s3_df["train_w_metric"] * s3_df["train_L_metric_multiscale"],
    "log": s3_df["train_w_log"] * s3_df["train_L_log"],
    "sparse": s3_df["train_w_sparse"] * s3_df["train_L_sparse_pre_anchor"],
    "edge": s3_df["train_w_edge"] * s3_df["train_L_edge"],
}
best_row=s3_df.loc[s3_df["val_rmse"].idxmin()]
budget=[]
for name in loss_parts:
    value=float(loss_parts[name].loc[best_row.name]); budget.append({"component":name,"weighted_loss":value})
budget_df=pd.DataFrame(budget); budget_df["share_pct"]=100*budget_df["weighted_loss"]/budget_df["weighted_loss"].sum()
budget_df.to_csv(RUN_LOCAL / "s3_loss_budget.csv",index=False); display(budget_df)

gate_columns=[c for c in s3_df.columns if c.startswith("train_mean_residual_gate_")]
gate_df=s3_df[["epoch",*gate_columns]].copy(); gate_df.to_csv(RUN_LOCAL / "s3_gate_dynamics.csv",index=False)
for column in gate_columns:
    axes[2].plot(gate_df["epoch"],gate_df[column],label=column.replace("train_mean_residual_gate_","gate "))
axes[2].axhline(.05,color="black",ls="--",lw=1); axes[2].set(xlabel="epoch",ylabel="mean gate",title="Residual gates"); axes[2].legend()
fig.tight_layout(); fig.savefig(RUN_LOCAL / "s3_vs_s2_benchmark.png",dpi=160,bbox_inches="tight"); plt.show()
subprocess.run(["rsync","-a",f"{RUN_LOCAL}/",f"{RUN_DRIVE}/"],check=True)

## 9. Verify canonical outputs

Require validation metrics, 1,000 test PNGs, FP16 profile, and the canonical experiment record.

In [ ]:
# 10) Verify canonical source of truth and sync final artifacts
metric_file = RUN_LOCAL / "logs" / "infer_val_metrics_global.json"
assert metric_file.is_file(), metric_file
print(metric_file.read_text())

test_prediction_dir = (
    RUN_LOCAL
    / "test_predictions"
    / "benchmark_png"
)

test_png = list(test_prediction_dir.glob("*.png"))
assert len(test_png) == TEST_COUNT, len(test_png)

profile_file = (
    RUN_LOCAL
    / "logs"
    / "geolift_component_profile.json"
)

assert profile_file.is_file(), profile_file
canonical_record = json.loads((RUN_LOCAL / "experiment_record.json").read_text(encoding="utf-8"))
assert canonical_record["status"] == "complete"
assert canonical_record["protocol"]["lock"]["protocol_sha256"] == canonical_record["protocol"]["protocol_sha256"]

subprocess.run(
    [
        "rsync",
        "-a",
        f"{RUN_LOCAL}/",
        f"{RUN_DRIVE}/",
    ],
    check=True,
)

print("Validation metrics:", metric_file)
print("KITTI ZIP:", RUN_LOCAL / "kitti_test_predictions.zip")
print("Runtime profile:", profile_file)
print("Source of truth:", RUN_LOCAL / "experiment_record.json")
print("Saved final run to:", RUN_DRIVE)


## Drive output

```text
MyDrive/GeoLift_RT_runs/v3_s3_lite_pretrained_train1600_val400/
|-- checkpoints/{best.pth,last.pth,epoch_*.pth}
|-- logs/{train_log.csv,train_log.jsonl,train_student.log}
|-- logs/{infer_val_metrics_global.json,geolift_component_profile.json}
|-- s3_vs_s2_metrics.csv
|-- s3_vs_s2_efficiency.csv
|-- s3_loss_budget.csv
|-- s3_gate_dynamics.csv
|-- s3_vs_s2_benchmark.png
|-- experiment_record.json
`-- kitti_test_predictions.zip
```

`RESUME_S3=True` resumes only an S3 checkpoint. For a clean epoch-0 benchmark, use a new output directory or move/delete the previous S3 run directory on Drive.